# Run Tau-Bench (Agentic Benchmark) with the SageMaker Inspect AI Container

## What is Tau-Bench?

[Tau-Bench](https://github.com/sierra-research/tau2-bench) (Tau2) is an agentic benchmark that evaluates language models on realistic, multi-turn tool-use customer service scenarios. The model must use tools, follow policies, and complete tasks across multiple conversation turns.

### Domains

| Domain | Description | Scoring |
|--------|-------------|---------|
| `tau2_airline` | Airline customer service (booking changes, cancellations) | Tool calls + goal assessment via grading model |
| `tau2_retail` | Retail customer service (orders, returns) | Tool calls + information relay verification |
| `tau2_telecom` | Telecom customer service (plans, billing) | Tool calls + end state comparison |

## How This Notebook Works

This notebook runs tau-bench as a **SageMaker Training Job** using the [SageMaker Inspect AI container](https://inspect.ai-safety-institute.org.uk/). The Training Job is used as managed compute for orchestration (not for model training) — your model runs on its own SageMaker inference endpoint.

## ⚠️ Token Usage Warning

Tau-bench uses a **high amount of tokens** (~60M for airline domain on a full run). Use `limit` and `message_limit` parameters to control costs during testing.

## Prerequisites

1. **AWS credentials configured** — via `aws configure`, environment variables, or IAM role
2. **A running SageMaker endpoint** — serving an OpenAI Chat Completions-compatible API with tool calling support
3. **S3 bucket** — for storing eval config and results
4. **IAM execution role** — with `sagemaker:InvokeEndpoint`, `AmazonS3FullAccess`, and `AmazonSageMakerFullAccess` permissions
5. **SageMaker training instance quota** — `ml.m5.large` (available by default in most accounts)

**Time to complete:** ~10-30 minutes depending on sample limit

**Cost:** Only the `ml.m5.large` orchestrator instance (~$0.13/hr) + your existing endpoint costs

## Step 1: Install Dependencies and Verify Credentials

In [ ]:
# Install dependencies (restart kernel after if needed)
!pip install "boto3" "sagemaker" pyyaml --quiet

# Verify sagemaker installed correctly
from importlib.metadata import version
print(f"sagemaker SDK version: {version('sagemaker')}")

In [ ]:
import boto3
import json
import os
import time
import yaml

# Verify AWS credentials
try:
    sts = boto3.client("sts")
    identity = sts.get_caller_identity()
    print(f"✓ AWS credentials configured")
    print(f"  Account: {identity['Account']}")
    print(f"  User/Role: {identity['Arn']}")
except Exception as e:
    print(f"✗ AWS credentials not configured: {e}")

## Step 2: Configure Your Environment

Set the region, endpoint name, S3 bucket, and IAM role.

**Important:** Set `REGION` to the region where your endpoint is deployed.

In [ ]:
# =============================================================================
# CONFIGURATION — Update these values
# =============================================================================

# Region where your endpoint is deployed
REGION = "us-east-1" # <-- REPLACE THIS

# Name of your existing SageMaker endpoint
ENDPOINT_NAME = "YOUR_ENDPOINT"  # <-- REPLACE THIS

# =============================================================================
# Auto-configured (no changes needed below)
# =============================================================================
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
S3_BUCKET = f"inspect-tau-bench-{ACCOUNT_ID}"
ROLE_NAME = "InspectLensEvalRole"
ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"

CONTAINER_IMAGES = {
    "us-east-1": "763104351884.dkr.ecr.us-east-1.amazonaws.com/sagemaker-inspect-ai:latest",
    "us-west-2": "763104351884.dkr.ecr.us-west-2.amazonaws.com/sagemaker-inspect-ai:latest",
    "eu-west-2": "763104351884.dkr.ecr.eu-west-2.amazonaws.com/sagemaker-inspect-ai:latest",
}
IMAGE_URI = CONTAINER_IMAGES[REGION]

sagemaker_client = boto3.client("sagemaker", region_name=REGION)

print(f"Configuration:")
print(f"  Region: {REGION}")
print(f"  Endpoint: {ENDPOINT_NAME}")
print(f"  Account: {ACCOUNT_ID}")
print(f"  S3 Bucket: {S3_BUCKET}")
print(f"  Role: {ROLE_ARN}")
print(f"  Image: {IMAGE_URI}")

### Verify Your Endpoint is Running

In [ ]:
try:
    response = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = response["EndpointStatus"]
    if status == "InService":
        print(f"✓ Endpoint '{ENDPOINT_NAME}' is InService")
    else:
        print(f"⚠ Endpoint '{ENDPOINT_NAME}' status: {status}")
        print("  Wait for it to reach InService before proceeding.")
except sagemaker_client.exceptions.ClientError as e:
    if "Could not find endpoint" in str(e):
        print(f"✗ Endpoint '{ENDPOINT_NAME}' not found in region {REGION}.")
        print(f"  To list endpoints: aws sagemaker list-endpoints --region {REGION}")
    else:
        raise

## Step 3: Create IAM Role and S3 Bucket

The SageMaker Training Job runs under an execution role. This role needs permissions to:
- Invoke your SageMaker endpoint (`sagemaker:InvokeEndpoint`)
- Read config and write results to S3 (`AmazonS3FullAccess`)
- Create and manage training jobs (`AmazonSageMakerFullAccess`)

In [ ]:
iam = boto3.client("iam")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    print(f"✓ Created role: {ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"✓ Role already exists: {ROLE_NAME}")

policies = [
    "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
    "arn:aws:iam::aws:policy/AmazonS3FullAccess",
]
for arn in policies:
    iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=arn)
    print(f"  Attached: {arn.split("/")[-1]}")

print("Waiting 10s for role propagation...")
time.sleep(10)

In [ ]:
# Create S3 bucket for config and results
s3 = boto3.client("s3", region_name=REGION)

try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=S3_BUCKET)
    else:
        s3.create_bucket(Bucket=S3_BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})
    print(f"✓ Created bucket: {S3_BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"✓ Bucket already exists: {S3_BUCKET}")

## Step 4: Create and Upload Benchmark

The container needs a benchmark Python file on S3 that defines the tau-bench task.
This file imports tau2 from `inspect-evals` (which is pre-installed in the container).

In [ ]:
os.makedirs("benchmarks", exist_ok=True)

# Create one benchmark file per domain (filename must match task name in config)
domains = {
    "tau2_airline": "from inspect_evals.tau2 import tau2_airline as _t\nfrom inspect_ai import task\n\n@task\ndef tau2_airline(**kwargs):\n    return _t(**kwargs)\n",
    "tau2_retail": "from inspect_evals.tau2 import tau2_retail as _t\nfrom inspect_ai import task\n\n@task\ndef tau2_retail(**kwargs):\n    return _t(**kwargs)\n",
    "tau2_telecom": "from inspect_evals.tau2 import tau2_telecom as _t\nfrom inspect_ai import task\n\n@task\ndef tau2_telecom(**kwargs):\n    return _t(**kwargs)\n",
}

for name, code in domains.items():
    with open(f"benchmarks/{name}.py", "w") as f:
        f.write(code)

with open("benchmarks/requirements.txt", "w") as f:
    f.write("inspect-evals\n")

# Upload to S3
s3 = boto3.client("s3", region_name=REGION)
for root, dirs, files in os.walk("benchmarks"):
    for filename in files:
        local_path = os.path.join(root, filename)
        s3.upload_file(local_path, S3_BUCKET, local_path)
        print(f"  Uploaded: s3://{S3_BUCKET}/{local_path}")

print("\n✓ Benchmark files uploaded")

## Step 5: Write the Eval Config

The container uses `inspect-evals` built-in tau2 tasks directly — no custom benchmark file upload needed.

### Key Parameters

| Parameter | Description | Recommended |
|-----------|-------------|-------------|
| `tasks` | Which tau2 domain(s) to evaluate | `tau2_airline`, `tau2_retail`, `tau2_telecom` |
| `limit` | Number of samples per task | 5 for testing, omit for full |
| `message_limit` | Max messages per sample (controls token usage) | 10 for testing, 100 for full |
| `max_connections` | Parallel requests to endpoint | 4-16 for single instance |
| `max_retries` | Retry attempts for transient errors | 10-50 |

In [ ]:
config = {
    "inference_provider": {
        "sagemaker_endpoint": {
            "endpoint_name": ENDPOINT_NAME,
            "region": REGION,
        }
    },
    "benchmarks": {
        "s3_path": f"s3://{S3_BUCKET}/benchmarks/",
        "tasks": [
            # Uncomment the domains you want to evaluate:
            {"name": "tau2_airline", "limit": 5, "task_args": {"message_limit": 10}},
            # {"name": "tau2_retail", "limit": 5, "task_args": {"message_limit": 10}},
            # {"name": "tau2_telecom", "limit": 5, "task_args": {"message_limit": 10}},
        ],
    },
    "eval": {
        "max_connections": 4,
        "max_retries": 10,
        "timeout": 600,
        "decoding": {
            "temperature": 0.0,
            "max_tokens": 8192,
        },
    },
    "output": {
        "s3_path": f"s3://{S3_BUCKET}/eval-results/",
    },
}

os.makedirs("config", exist_ok=True)
with open("config/inspect_config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

s3.upload_file("config/inspect_config.yaml", S3_BUCKET, "config/inspect_config.yaml")

print("✓ Config uploaded to S3. Contents:")
print("---")
print(yaml.dump(config, default_flow_style=False))

## Step 6: Submit the Evaluation Job

The training job instance (`ml.m5.large`) acts as the orchestrator — it sends requests to your endpoint and collects results.

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import InputData, Compute
from sagemaker.core.shapes.shapes import StoppingCondition, OutputDataConfig
from sagemaker.core.helper.session_helper import Session as SMSession

# Pin the sagemaker session to the correct region
sm_session = SMSession(boto_session=boto3.Session(region_name=REGION))

trainer = ModelTrainer(
    training_image=IMAGE_URI,
    role=ROLE_ARN,
    compute=Compute(
        instance_type="ml.m5.large",
        instance_count=1,
        volume_size_in_gb=30,
    ),
    output_data_config=OutputDataConfig(
        s3_output_path=f"s3://{S3_BUCKET}/output/",
    ),
    stopping_condition=StoppingCondition(
        max_runtime_in_seconds=86400,
    ),
    base_job_name="inspect-tau-bench",
    sagemaker_session=sm_session,
)

trainer.train(
    input_data_config=[
        InputData(
            channel_name="config",
            data_source=f"s3://{S3_BUCKET}/config/",
        )
    ],
    wait=False,
)

job_name = trainer._latest_training_job.training_job_name
print(f"✓ Job submitted: {job_name}")
print(f"Monitor in console:")
print(f"  https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{job_name}")

## Step 7: Monitor Progress

Tau-bench evaluations take longer than standard benchmarks due to multi-turn conversations.
Expect ~10-30 minutes for 5 samples with `message_limit=10`.

> **If the job fails**, check:
> - The endpoint name is correct and is `InService`
> - The endpoint supports tool calling (OpenAI Chat Completions format)
> - The IAM role has `sagemaker:InvokeEndpoint` permission

In [ ]:
print(f"Waiting for job '{job_name}' to complete...")
print()

while True:
    response = sagemaker_client.describe_training_job(TrainingJobName=job_name)
    status = response["TrainingJobStatus"]
    secondary = response.get("SecondaryStatus", "")

    if status in ("Completed", "Failed", "Stopped"):
        print(f"\n✓ Job finished: {status}")
        if status == "Failed":
            print(f"  Reason: {response.get('FailureReason', 'Unknown')}")
        break

    print(f"  {status} / {secondary}...", end="\r")
    time.sleep(15)

## Step 8: View Results

In [ ]:
# Download results from S3
os.makedirs("results", exist_ok=True)

prefix = f"eval-results/{job_name}/eval_results/"
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix)

if "Contents" in response:
    for obj in response["Contents"]:
        key = obj["Key"]
        local_path = os.path.join("results", os.path.basename(key))
        s3.download_file(S3_BUCKET, key, local_path)
        print(f"  Downloaded: {os.path.basename(key)} ({obj['Size']} bytes)")
else:
    print("No results found. Check job status above.")
    print(f"  Looked in: s3://{S3_BUCKET}/{prefix}")

In [ ]:
# View evaluation status summary
try:
    with open("results/_status.json") as f:
        status = json.load(f)
    print("Evaluation Status:")
    print(f"  Status: {status['status']}")
    print(f"  Passed: {status['passed_tasks']}")
    print(f"  Failed: {status['failed_tasks']}")
    if status["failure_reasons"]:
        print(f"  Reasons: {status['failure_reasons']}")
except FileNotFoundError:
    print("Status file not found.")

In [ ]:
# View detailed scores
try:
    from inspect_ai.log import read_eval_log
    eval_files = [f for f in os.listdir("results") if f.endswith(".eval")]
    for eval_file in eval_files:
        log = read_eval_log(os.path.join("results", eval_file))
        print(f"Task: {log.eval.task}")
        print(f"Model: {log.eval.model}")
        print(f"Samples: {len(log.samples) if log.samples else 0}")
        if log.results and log.results.scores:
            for score in log.results.scores:
                for name, metric in score.metrics.items():
                    print(f"  {name}: {metric.value}")
        print()
except ImportError:
    print("Install inspect-ai to view detailed scores: %pip install inspect-ai")
except Exception as e:
    print(f"Could not read eval log: {e}")

### View with Inspect AI Viewer

For interactive exploration of per-sample conversation traces and scoring:

In [ ]:
%pip install inspect-ai --quiet

!INSPECT_LOG_DIR=./results inspect view

### Interpreting Results

| Observation | Meaning |
|-------------|---------|
| Many errors (E) | Endpoint compatibility issue — check tool calling support |
| Failures with wrong tool calls | Model needs better tool-calling fine-tuning |
| Failures with correct tools but unmet goals | Model struggles with policy/reasoning |

### Reference Results (GPT-5, pass^1 rate)

| Domain | Accuracy | Stderr | Tau2 Leaderboard |
|--------|----------|--------|------------------|
| Retail | 0.825 | 0.036 | 0.816 |
| Airline | 0.580 | 0.071 | 0.625 |
| Telecom | 0.939 | 0.023 | 0.958 |

*Leaderboard uses 4 epochs and message_limit=100.*

## Cleanup

Remove eval artifacts from S3. **This does NOT delete your endpoint or IAM role.**

In [ ]:
# Delete S3 objects and bucket
s3 = boto3.client("s3", region_name=REGION)
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=S3_BUCKET):
    if "Contents" in page:
        objects = [{"Key": obj["Key"]} for obj in page["Contents"]]
        s3.delete_objects(Bucket=S3_BUCKET, Delete={"Objects": objects})
s3.delete_bucket(Bucket=S3_BUCKET)
print(f"✓ Deleted bucket: {S3_BUCKET}")

# Detach policies and delete role
iam = boto3.client("iam")
for arn in policies:
    iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=arn)
iam.delete_role(RoleName=ROLE_NAME)
print(f"✓ Deleted role: {ROLE_NAME}")

# Clean up local files
import shutil
for d in ["config", "results"]:
    shutil.rmtree(d, ignore_errors=True)
print("✓ Local files cleaned up.")
print("Note: Your SageMaker endpoint is still running. Delete it separately if no longer needed.")

## Next Steps

1. **Run the full benchmark**: Remove `limit` and increase `message_limit` to 100
2. **Try all domains**: Uncomment `tau2_retail` and `tau2_telecom` in the config
3. **Scale up**: Increase `max_connections` for multi-instance endpoints
4. **Multiple epochs**: Add `epochs: 4` to match the official leaderboard methodology
5. **Compare models**: Run the same config against different endpoints

### Resources

- [Inspect AI Documentation](https://inspect.ai-safety-institute.org.uk/)
- [Inspect Evals Repository](https://github.com/UKGovernmentBEIS/inspect_evals)
- [Tau2 Paper](https://arxiv.org/abs/2506.07982)
- [Tau2 Leaderboard](https://taubench.com/#leaderboard)
- [AWS SageMaker Documentation](https://docs.aws.amazon.com/sagemaker/)